# Waterbirds — Passo 2: quatro arquiteturas leves

A ResNet-50 do Passo 1 completa a comparação das cinco arquiteturas. Este notebook roda **ShuffleNet V2 ×0,5, MobileNetV3-Small, DenseNet-121 e EfficientNet-B0**. Ele usa o Waterbirds oficial e mantém as 100 épocas, ERM, DFR e as métricas da etapa de reprodução.

1. Selecione GPU no Colab (por exemplo, T4). As três primeiras células de código verificam a GPU, instalam o ambiente e baixam os dados oficiais.
2. A célula de configuração começa com **uma semente**. Rode somente a célula da ShuffleNet para confirmar que o experimento completo da semente termina. Uma semente é apenas uma verificação preliminar.
3. Para obter os resultados finais, troque para `[1, 2, 3, 4, 5]`. Rode **uma arquitetura por vez**. Uma semente já concluída é reaproveitada se os arquivos continuarem disponíveis nessa sessão, com o mesmo ambiente.
4. Depois de cada arquitetura ou bloco de sementes, execute a célula **Baixar**. Se a sessão for desconectada e os arquivos forem perdidos, guarde os ZIPs das sementes concluídas e rode somente o que falta em outra sessão compatível. Os arquivos serão úteis para juntar os resultados por semente.

Você pode usar a EfficientNet-B0 do plano anterior se ela foi treinada e avaliada com o mesmo protocolo. Ela não precisa ser refeita.


In [ ]:
import csv
import hashlib
import os
import pathlib
import shutil
import subprocess
import sys
import tarfile
import zipfile

ROOT = pathlib.Path("/content/waterbirds-passo-2")
ROOT.mkdir(parents=True, exist_ok=True)

def run(*args):
    return subprocess.run([str(arg) for arg in args], cwd=ROOT, check=True)

run("nvidia-smi", "--query-gpu=name,driver_version", "--format=csv,noheader")

VENV = ROOT / ".venv-py312"
PYTHON = VENV / "bin" / "python"

if not PYTHON.exists():
    uv = shutil.which("uv")
    if uv:
        uv_cmd = [uv]
    else:
        run(sys.executable, "-m", "pip", "install", "-q", "uv")
        uv_cmd = [sys.executable, "-m", "uv"]

    run(*uv_cmd, "venv", "--python", "3.12", "--seed", VENV)

run(PYTHON, "--version")
run(
    PYTHON, "-m", "pip", "install",
    "torch==2.5.1", "torchvision==0.20.1",
    "--index-url", "https://download.pytorch.org/whl/cu124",
)
run(
    PYTHON, "-c",
    "import torch; assert torch.cuda.is_available(), 'CUDA indisponível'; "
    "print('GPU pronta:', torch.cuda.get_device_name(0))",
)


In [ ]:
# Cópia inspecionável dos arquivos locais da reprodução.
SOURCES = {'__main__.py': '"""Run the pinned Waterbirds protocol from the project root."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport py_compile\nimport shlex\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nfrom importlib import metadata\nfrom pathlib import Path\n\nfrom .metrics import REFERENCE, read_metadata, result_rows, summarize, write_csv\n\nHERE = Path(__file__).resolve().parent\nUPSTREAM = "9d8d5de78feafb0ba99878b8a1f91c5cd942e62a"\nARCHITECTURES = ("resnet50", "densenet121", "efficientnet_b0", "convnext_tiny", "vit_b_16",\n                 "shufflenet_v2_x0_5", "mobilenet_v3_small")\nSEEDS = (1, 2, 3, 4, 5)\nUPSTREAM_FILES = ("LICENSE", "train_classifier.py", "dfr_evaluate_spurious.py", "wb_data.py", "utils.py")\n\n\ndef prepare(upstream, destination):\n    for name in UPSTREAM_FILES:\n        blob = subprocess.check_output(["git", "-C", str(upstream), "show", f"{UPSTREAM}:{name}"])\n        (destination / name).write_bytes(blob)\n    patch = HERE / "upstream.patch"\n    subprocess.run(["git", "apply", "--whitespace=nowarn", str(patch)], cwd=destination, check=True)\n    shutil.copyfile(HERE / "architectures.py", destination / "waterbirds_models.py")\n    for script in destination.glob("*.py"):\n        py_compile.compile(str(script), doraise=True)\n\n\ndef validate_data(data):\n    path = data / "metadata.csv"\n    info = read_metadata(path)\n    with path.open(newline="", encoding="utf-8") as handle:\n        for row in csv.DictReader(handle):\n            if not (data / row["img_filename"]).is_file():\n                raise ValueError(f"Imagem ausente: {row[\'img_filename\']}")\n    return info\n\n\ndef commands(work, data, out, architecture, seed):\n    common = ["--architecture", architecture, "--data_dir", str(data), "--seed", str(seed)]\n    return (\n        [sys.executable, str(work / "train_classifier.py"), *common, "--test_wb_dir", str(data),\n         "--output_dir", str(out / "erm"), "--pretrained_model", "--augment_data",\n         "--num_epochs", "100", "--weight_decay", "0.001", "--batch_size", "32",\n         "--momentum_decay", "0.9", "--init_lr", "0.001", "--eval_freq", "1"],\n        [sys.executable, str(work / "dfr_evaluate_spurious.py"), *common,\n         "--ckpt_path", str(out / "erm" / "final_checkpoint.pt"),\n         "--result_path", str(out / "raw.pending.json")],\n    )\n\n\ndef write_reports(out, dataset, architecture):\n    rows, finished = [], []\n    for seed in SEEDS:\n        raw = out / f"seed-{seed}" / "raw.json"\n        if raw.is_file():\n            rows.extend(result_rows(json.loads(raw.read_text()), dataset["group_counts"], architecture, seed))\n            finished.append(seed)\n    write_csv(out / "results.csv", rows)\n    if tuple(finished) != SEEDS:\n        print(f"Sementes finalizadas: {finished}. O resumo exige {list(SEEDS)}.")\n        return\n    summary = summarize(rows, SEEDS)\n    write_csv(out / "summary.csv", summary)\n    if architecture == "resnet50":\n        comparison = []\n        for row in summary:\n            for metric, (published_mean, published_sd) in REFERENCE[row["method"]].items():\n                observed = row[f"{metric}_mean"]\n                comparison.append({\n                    "method": row["method"], "metric": metric, "n": row["n"],\n                    "paper_mean_pct": published_mean, "paper_sd_pp": published_sd,\n                    "observed_mean_pct": observed, "observed_sd_pp": row[f"{metric}_sd"],\n                    "delta_pp": observed - published_mean,\n                })\n        write_csv(out / "paper_comparison.csv", comparison)\n\n\ndef run(work, data, output_root, dataset, architecture, seeds):\n    import torch\n\n    from .architectures import MODEL_SPECS\n\n    if not torch.cuda.is_available():\n        raise ValueError("Treino de referência requer CUDA. Use check neste computador e run em uma GPU NVIDIA.")\n    out = output_root / architecture\n    out.mkdir(parents=True, exist_ok=True)\n    manifest = {\n        **dataset,\n        "profile": "kirichenko-iclr2023-waterbirds-table2",\n        "architecture": architecture,\n        "pretrained_weights": str(MODEL_SPECS[architecture][1]),\n        "upstream_commit": UPSTREAM,\n        "patch_sha256": hashlib.sha256((HERE / "upstream.patch").read_bytes()).hexdigest(),\n        "model_adapter_sha256": hashlib.sha256((HERE / "architectures.py").read_bytes()).hexdigest(),\n        "dependencies": {name: metadata.version(name) for name in (\n            "torch", "torchvision", "numpy", "pandas", "scikit-learn", "pillow", "tensorboard")},\n        "python": sys.version.split()[0],\n        "cuda": torch.version.cuda,\n        "gpu": torch.cuda.get_device_name(),\n        "seeds": SEEDS,\n    }\n    contents = json.dumps(manifest, indent=2, sort_keys=True) + "\\n"\n    manifest_path = out / "protocol.json"\n    if manifest_path.exists() and json.loads(manifest_path.read_text()) != json.loads(contents):\n        raise ValueError("A saída contém resultados com outro protocolo/ambiente. Use outro --output.")\n    if not manifest_path.exists() and any(out.iterdir()):\n        raise ValueError("A pasta já contém arquivos sem protocolo registrado. Use outro --output.")\n    manifest_path.write_text(contents)\n    for seed in seeds:\n        seed_dir = out / f"seed-{seed}"\n        raw = seed_dir / "raw.json"\n        if raw.exists():\n            print(f"Semente {seed} já concluída: {raw}", flush=True)\n            continue\n        seed_dir.mkdir(exist_ok=True)\n        train, evaluate = commands(work, data, seed_dir, architecture, seed)\n        with (seed_dir / "commands.txt").open("w") as handle:\n            for command in (train, evaluate):\n                handle.write(shlex.join(command) + "\\n")\n        # A finished ERM checkpoint is reusable if only DFR was interrupted.\n        if not (seed_dir / "erm" / "final_checkpoint.pt").is_file():\n            subprocess.run(train, cwd=work, check=True)\n        pending = seed_dir / "raw.pending.json"\n        pending.unlink(missing_ok=True)\n        print(f"Avaliando ERM e DFR — semente {seed}. Log: {seed_dir / \'evaluation.log\'}", flush=True)\n        with (seed_dir / "evaluation.log").open("w") as log:\n            subprocess.run(evaluate, cwd=work, stdout=log, stderr=subprocess.STDOUT, check=True)\n        result_rows(json.loads(pending.read_text()), dataset["group_counts"], architecture, seed)\n        pending.replace(raw)\n        write_reports(out, dataset, architecture)\n    write_reports(out, dataset, architecture)\n    print(f"Resultados: {out / \'results.csv\'}")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("action", choices=("check", "run"))\n    parser.add_argument("--data", type=Path, default=Path("data/waterbirds_v1.0"))\n    parser.add_argument("--upstream", type=Path, default=Path(".upstream/deep_feature_reweighting"))\n    parser.add_argument("--output", type=Path, default=Path("runs"))\n    parser.add_argument("--architecture", choices=ARCHITECTURES, default="resnet50")\n    parser.add_argument("--seed", type=int, choices=SEEDS, action="append",\n                        help="Executar só esta semente; repetível. Padrão: 1 a 5.")\n    args = parser.parse_args()\n    try:\n        dataset = validate_data(args.data.resolve())\n        with tempfile.TemporaryDirectory(prefix="waterbirds-dfr-") as folder:\n            work = Path(folder)\n            prepare(args.upstream.resolve(), work)\n            if args.action == "check":\n                print(json.dumps({**dataset, "upstream_commit": UPSTREAM,\n                                  "status": "dados, patch e sintaxe verificados; nenhum treino iniciado"}, indent=2))\n                return\n            run(work, args.data.resolve(), args.output.resolve(), dataset,\n                args.architecture, sorted(set(args.seed or SEEDS)))\n    except (OSError, ValueError, KeyError, ImportError, subprocess.CalledProcessError) as error:\n        parser.exit(1, f"Erro: {error}\\n")\n\n\nif __name__ == "__main__":\n    main()\n', 'architectures.py': '"""Model adapter copied next to the pinned upstream scripts at runtime."""\nfrom functools import reduce\n\nimport torch\nfrom torchvision import models\n\n# Use explicit weight enums; DEFAULT may silently change between versions.\nMODEL_SPECS = {\n    "resnet50": (models.resnet50, models.ResNet50_Weights.IMAGENET1K_V1, ("fc",)),\n    "densenet121": (models.densenet121, models.DenseNet121_Weights.IMAGENET1K_V1, ("classifier",)),\n    "efficientnet_b0": (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1, ("classifier", "1")),\n    "convnext_tiny": (models.convnext_tiny, models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1, ("classifier", "2")),\n    "vit_b_16": (models.vit_b_16, models.ViT_B_16_Weights.IMAGENET1K_V1, ("heads", "head")),\n    "shufflenet_v2_x0_5": (models.shufflenet_v2_x0_5, models.ShuffleNet_V2_X0_5_Weights.IMAGENET1K_V1, ("fc",)),\n    "mobilenet_v3_small": (models.mobilenet_v3_small, models.MobileNet_V3_Small_Weights.IMAGENET1K_V1, ("classifier", "3")),\n}\n\n\ndef make_model(architecture, n_classes, pretrained):\n    build, weights, path = MODEL_SPECS[architecture]\n    model = build(weights=weights if pretrained else None)\n    parent = reduce(getattr, path[:-1], model)\n    previous = getattr(parent, path[-1])\n    head = torch.nn.Linear(previous.in_features, n_classes)\n    # torchvision\'s ViT initializes its replacement head to zero before training.\n    if architecture == "vit_b_16":\n        torch.nn.init.zeros_(head.weight)\n        torch.nn.init.zeros_(head.bias)\n    setattr(parent, path[-1], head)\n    return model\n\n\ndef get_embedding(model, images, architecture):\n    head = reduce(getattr, MODEL_SPECS[architecture][2], model)\n    features = []\n    hook = head.register_forward_pre_hook(lambda _module, args: features.append(args[0]))\n    try:\n        model(images)\n    finally:\n        hook.remove()\n    return features[0]\n', 'metrics.py': '"""Waterbirds metadata and shared evaluation contract (percentages, not fractions)."""\nimport csv\nimport hashlib\nimport math\nfrom collections import defaultdict\nfrom statistics import mean, stdev\n\nOFFICIAL_COUNTS = {\n    "train": (3498, 184, 56, 1057),\n    "val": (467, 466, 133, 133),\n    "test": (2255, 2255, 642, 642),\n}\nGROUP_IDS = ("0_0", "0_1", "1_0", "1_1")\nSCORES = ("worst_group_accuracy_pct", "weighted_accuracy_pct", "micro_accuracy_pct",\n          "group_gap_pp", *(f"g{i}_accuracy_pct" for i in range(4)))\nREFERENCE = {\n    "ERM": {"worst_group_accuracy_pct": (74.9, 2.4), "weighted_accuracy_pct": (98.1, 0.1)},\n    "DFR": {"worst_group_accuracy_pct": (92.9, 0.2), "weighted_accuracy_pct": (94.2, 0.4)},\n}\n\n\ndef read_metadata(path):\n    counts = {split: [0] * 4 for split in OFFICIAL_COUNTS}\n    seen = set()\n    with path.open(newline="", encoding="utf-8") as handle:\n        for row in csv.DictReader(handle):\n            name = row["img_filename"]\n            if not name or name in seen:\n                raise ValueError(f"Imagem vazia ou repetida nos metadados: {name!r}")\n            seen.add(name)\n            split, label, place = row["split"], row["y"], row["place"]\n            if split not in ("0", "1", "2") or label not in ("0", "1") or place not in ("0", "1"):\n                raise ValueError(f"Rótulo, fundo ou divisão inválida: {name}")\n            counts[("train", "val", "test")[int(split)]][2 * int(label) + int(place)] += 1\n    counts = {split: tuple(values) for split, values in counts.items()}\n    if counts != OFFICIAL_COUNTS:\n        raise ValueError(f"Contagens incompatíveis com as divisões oficiais: {counts}")\n    return {"metadata_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),\n            "group_counts": counts}\n\n\ndef result_rows(raw, counts, architecture, seed):\n    for split in ("train", "test"):\n        if len(counts[split]) != 4 or min(counts[split]) <= 0:\n            raise ValueError(f"{split} precisa conter os quatro grupos.")\n    result = []\n    for method, values in (\n        ("ERM", [raw["erm"][f"accuracy_{group}"] for group in GROUP_IDS]),\n        ("DFR", raw["dfr"]["test_accs"]),\n    ):\n        if len(values) != 4 or not all(math.isfinite(v) and 0 <= v <= 1 for v in values):\n            raise ValueError("Esperadas quatro acurácias entre 0 e 1.")\n        groups = [v * 100 for v in values]\n        result.append({\n            "architecture": architecture, "method": method, "seed": seed,\n            "worst_group_accuracy_pct": min(groups),\n            "weighted_accuracy_pct": sum(a * n for a, n in zip(groups, counts["train"])) / sum(counts["train"]),\n            "micro_accuracy_pct": sum(a * n for a, n in zip(groups, counts["test"])) / sum(counts["test"]),\n            "group_gap_pp": max(groups) - min(groups),\n            **{f"g{i}_accuracy_pct": value for i, value in enumerate(groups)},\n            "dfr_c": raw["dfr"]["best_hypers"][0] if method == "DFR" else "",\n        })\n    return result\n\n\ndef summarize(rows, expected_seeds):\n    pairs = defaultdict(dict)\n    for row in rows:\n        runs = pairs[row["architecture"], row["method"]]\n        if row["seed"] in runs:\n            raise ValueError("Combinação de arquitetura, método e semente duplicada.")\n        runs[row["seed"]] = row\n    if not pairs:\n        raise ValueError("Sem resultados para resumir.")\n    summary = []\n    for (architecture, method), runs in sorted(pairs.items()):\n        if set(runs) != set(expected_seeds):\n            raise ValueError(f"Conjunto de sementes incompleto ou diferente: {architecture}/{method}")\n        entry = {"architecture": architecture, "method": method, "n": len(runs)}\n        for score in SCORES:\n            values = [row[score] for row in runs.values()]\n            entry[f"{score}_mean"] = mean(values)\n            entry[f"{score}_sd"] = stdev(values) if len(values) > 1 else ""\n        summary.append(entry)\n    if any({method for arch, method in pairs if arch == a} != {"ERM", "DFR"} for a, _ in pairs):\n        raise ValueError("Cada arquitetura precisa dos dois métodos: ERM e DFR.")\n    return summary\n\n\ndef write_csv(path, rows):\n    with path.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n', 'requirements.txt': '# Instale torch==2.5.1 e torchvision==0.20.1 separadamente (veja o README).\nnumpy==1.26.4\npandas==2.2.3\nscikit-learn==1.5.2\nPillow==11.0.0\ntensorboard==2.18.0\ntqdm==4.67.1\n', 'upstream.patch': 'diff --git a/dfr_evaluate_spurious.py b/dfr_evaluate_spurious.py\nindex 5af70a4..84544e4 100644\n--- a/dfr_evaluate_spurious.py\n+++ b/dfr_evaluate_spurious.py\n@@ -12,7 +12,7 @@ import sys\n from collections import defaultdict\n import json\n from functools import partial\n-import pickle\n+from waterbirds_models import MODEL_SPECS, make_model, get_embedding\n \n from sklearn.linear_model import LogisticRegression\n from sklearn.preprocessing import StandardScaler\n@@ -56,7 +56,11 @@ parser.add_argument(\n parser.add_argument(\n     "--tune_class_weights_dfr_train", action=\'store_true\',\n     help="Learn class weights for DFR(Train)")\n+parser.add_argument("--architecture", choices=MODEL_SPECS, default="resnet50")\n+parser.add_argument("--device", choices=("cuda", "cpu"), default="cuda")\n+parser.add_argument("--seed", type=int, default=1)\n args = parser.parse_args()\n+set_seed(args.seed)\n \n \n def dfr_on_validation_tune(\n@@ -319,7 +323,7 @@ valset = WaterBirdsDataset(\n     basedir=args.data_dir, split="val", transform=test_transform)\n \n loader_kwargs = {\'batch_size\': args.batch_size,\n-                 \'num_workers\': 4, \'pin_memory\': True,\n+                 \'num_workers\': 4 if args.device == \'cuda\' else 0, \'pin_memory\': args.device == \'cuda\',\n                  "reweight_places": None}\n train_loader = get_loader(\n     trainset, train=True, reweight_groups=False, reweight_classes=False,\n@@ -333,13 +337,9 @@ val_loader = get_loader(\n \n # Load model\n n_classes = trainset.n_classes\n-model = torchvision.models.resnet50(pretrained=False)\n-d = model.fc.in_features\n-model.fc = torch.nn.Linear(d, n_classes)\n-model.load_state_dict(torch.load(\n-    args.ckpt_path\n-))\n-model.cuda()\n+model = make_model(args.architecture, n_classes, pretrained=False)\n+model.load_state_dict(torch.load(args.ckpt_path, map_location="cpu"))\n+model.to(args.device)\n model.eval()\n \n # Evaluate model\n@@ -355,22 +355,6 @@ print()\n model.eval()\n \n # Extract embeddings\n-def get_embed(m, x):\n-    x = m.conv1(x)\n-    x = m.bn1(x)\n-    x = m.relu(x)\n-    x = m.maxpool(x)\n-\n-    x = m.layer1(x)\n-    x = m.layer2(x)\n-    x = m.layer3(x)\n-    x = m.layer4(x)\n-\n-    x = m.avgpool(x)\n-    x = torch.flatten(x, 1)\n-    return x\n-\n-\n all_embeddings = {}\n all_y, all_p, all_g = {}, {}, {}\n for name, loader in [("train", train_loader), ("test", test_loader), ("val", val_loader)]:\n@@ -378,7 +362,7 @@ for name, loader in [("train", train_loader), ("test", test_loader), ("val", val\n     all_y[name], all_p[name], all_g[name] = [], [], []\n     for x, y, g, p in tqdm.tqdm(loader):\n         with torch.no_grad():\n-            all_embeddings[name].append(get_embed(model, x.cuda()).detach().cpu().numpy())\n+            all_embeddings[name].append(get_embedding(model, x.to(args.device), args.architecture).detach().cpu().numpy())\n             all_y[name].append(y.detach().cpu().numpy())\n             all_g[name].append(g.detach().cpu().numpy())\n             all_p[name].append(p.detach().cpu().numpy())\n@@ -397,7 +381,7 @@ c, w1, w2 = dfr_on_validation_tune(\n dfr_val_results["best_hypers"] = (c, w1, w2)\n print("Hypers:", (c, w1, w2))\n test_accs, test_mean_acc, train_accs = dfr_on_validation_eval(\n-        c, w1, w2, all_embeddings, all_y, all_g,\n+        c, w1, w2, all_embeddings, all_y, all_g, num_retrains=10,\n     balance_val=args.balance_dfr_val, add_train=not args.notrain_dfr_val)\n dfr_val_results["test_accs"] = test_accs\n dfr_val_results["train_accs"] = train_accs\n@@ -406,29 +390,6 @@ dfr_val_results["test_mean_acc"] = test_mean_acc\n print(dfr_val_results)\n print()\n \n-# DFR on train subsampled\n-print("DFR on train subsampled")\n-dfr_train_results = {}\n-c, w1, w2 = dfr_train_subset_tune(\n-    all_embeddings, all_y, all_g,\n-    learn_class_weights=args.tune_class_weights_dfr_train)\n-dfr_train_results["best_hypers"] = (c, w1, w2)\n-print("Hypers:", (c, w1, w2))\n-test_accs, test_mean_acc, train_accs = dfr_train_subset_eval(\n-        c, w1, w2, all_embeddings, all_y, all_g)\n-dfr_train_results["test_accs"] = test_accs\n-dfr_train_results["train_accs"] = train_accs\n-dfr_train_results["test_worst_acc"] = np.min(test_accs)\n-dfr_train_results["test_mean_acc"] = test_mean_acc\n-print(dfr_train_results)\n-print()\n-\n-\n-all_results = {}\n-all_results["base_model_results"] = base_model_results\n-all_results["dfr_val_results"] = dfr_val_results\n-all_results["dfr_train_results"] = dfr_train_results\n-print(all_results)\n-\n-with open(args.result_path, \'wb\') as f:\n-    pickle.dump(all_results, f)\n\\ No newline at end of file\n+with open(args.result_path, "w", encoding="utf-8") as f:\n+    json.dump({"erm": base_model_results["test"], "dfr": dfr_val_results}, f,\n+              default=lambda value: value.item(), allow_nan=False)\ndiff --git a/train_classifier.py b/train_classifier.py\nindex 2660bee..616a9b7 100644\n--- a/train_classifier.py\n+++ b/train_classifier.py\n@@ -13,7 +13,7 @@ from functools import partial\n \n from wb_data import WaterBirdsDataset, get_loader, get_transform_cub, log_data\n \n-from utils import MultiTaskHead\n+from waterbirds_models import MODEL_SPECS, make_model\n from utils import Logger, AverageMeter, set_seed, evaluate, get_y_p\n from utils import update_dict, get_results, write_dict_to_tb\n \n@@ -40,6 +40,8 @@ parser.add_argument(\n     default="logs/",\n     help="Output directory")\n \n+parser.add_argument("--architecture", choices=MODEL_SPECS, default="resnet50")\n+parser.add_argument("--device", choices=("cuda", "cpu"), default="cuda")\n parser.add_argument("--pretrained_model", action=\'store_true\', help="Use pretrained model")\n parser.add_argument("--reweight_classes", action=\'store_true\', help="Reweight classes")\n parser.add_argument("--reweight_places", action=\'store_true\', help="Reweight based on place")\n@@ -96,7 +98,6 @@ test_transform = get_transform_cub(target_resolution=target_resolution, train=Fa\n \n trainset = WaterBirdsDataset(basedir=basedir, split="train", transform=train_transform)\n testset_dict = {\n-    \'wb\': WaterBirdsDataset(basedir=args.test_wb_dir, split="test", transform=test_transform),\n     \'wb_val\': WaterBirdsDataset(basedir=args.test_wb_dir, split="val", transform=test_transform),\n }\n \n@@ -123,7 +124,7 @@ if args.num_minority_groups_remove > 0:\n # testset = WaterBirdsDataset(basedir=basedir, split="test", transform=test_transform)\n # valset = WaterBirdsDataset(basedir=basedir, split="val", transform=test_transform)\n \n-loader_kwargs = {\'batch_size\': args.batch_size, \'num_workers\': 4, \'pin_memory\': True}\n+loader_kwargs = {\'batch_size\': args.batch_size, \'num_workers\': 4 if args.device == \'cuda\' else 0, \'pin_memory\': args.device == \'cuda\'}\n train_loader = get_loader(\n     trainset, train=True, reweight_groups=args.reweight_groups,\n     reweight_classes=args.reweight_classes, reweight_places=args.reweight_places, **loader_kwargs)\n@@ -137,16 +138,12 @@ for test_name, testset_v in testset_dict.items():\n #   testset, train=False, reweight_groups=None, reweight_classes=None, **loader_kwargs)\n \n get_yp_func = partial(get_y_p, n_places=trainset.n_places)\n-log_data(logger, trainset, testset_dict[\'wb\'], get_yp_func=get_yp_func)\n \n # Model\n n_classes = trainset.n_classes\n-model = torchvision.models.resnet50(pretrained=args.pretrained_model)\n-d = model.fc.in_features\n-if not args.multitask:\n-    model.fc = torch.nn.Linear(d, n_classes)\n-else:\n-    model.fc = MultiTaskHead(d, [n_classes, trainset.n_places])\n+if args.multitask or args.predict_place or args.test_grey_dir or args.test_places_dir:\n+    parser.error("This protocol supports only Waterbirds bird classification.")\n+model = make_model(args.architecture, n_classes, args.pretrained_model)\n \n # TODO: fix resuming from a checkpoint\n if args.resume is not None:\n@@ -154,7 +151,7 @@ if args.resume is not None:\n     checkpoint = torch.load(args.resume)\n     model.load_state_dict(checkpoint)\n \n-model.cuda()\n+model.to(args.device)\n \n optimizer = torch.optim.SGD(\n     model.parameters(), lr=args.init_lr, momentum=args.momentum_decay, weight_decay=args.weight_decay)\n@@ -178,7 +175,7 @@ for epoch in range(args.num_epochs):\n \n     for batch in tqdm.tqdm(train_loader):\n         x, y, g, p = batch\n-        x, y, p = x.cuda(), y.cuda(), p.cuda()\n+        x, y, p = x.to(args.device), y.to(args.device), p.to(args.device)\n         if args.predict_place:\n             y = p\n \n@@ -241,4 +238,6 @@ for epoch in range(args.num_epochs):\n \n     logger.write(\'\\n\')\n \n-torch.save(model.state_dict(), os.path.join(args.output_dir, \'final_checkpoint.pt\'))\n+final_checkpoint = os.path.join(args.output_dir, \'final_checkpoint.pt\')\n+torch.save(model.state_dict(), final_checkpoint + \'.pending\')\n+os.replace(final_checkpoint + \'.pending\', final_checkpoint)\ndiff --git a/utils.py b/utils.py\nindex d22eb4d..932c231 100644\n--- a/utils.py\n+++ b/utils.py\n@@ -110,7 +110,8 @@ def evaluate(model, loader, get_yp_func, multitask=False, predict_place=False):\n \n     with torch.no_grad():\n         for x, y, g, p in tqdm.tqdm(loader):\n-            x, y, p = x.cuda(), y.cuda(), p.cuda()\n+            device = next(model.parameters()).device\n+            x, y, p = x.to(device), y.to(device), p.to(device)\n             if predict_place:\n                 y = p\n \n'}

pkg = ROOT / "reproduction"
pkg.mkdir(exist_ok=True)
for name, contents in SOURCES.items():
    (pkg / name).write_text(contents, encoding="utf-8")
run(PYTHON, "-m", "pip", "install", "-r", pkg / "requirements.txt")
upstream = ROOT / ".upstream" / "deep_feature_reweighting"
if not upstream.is_dir():
    run("git", "clone", "https://github.com/PolinaKirichenko/deep_feature_reweighting.git", upstream)


In [ ]:
download_urls = [
    "https://worksheets.codalab.org/rest/bundles/0x505056d5cdea4e4eaa0e242cbfe2daa4/contents/blob/",
    "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz",
]
expected_metadata_digest = "2f023b9dc371f6fbb82c7cb810ff0f03e19c54c8055a2e37517ae7bb56644362"
data_root = ROOT / "data"
data_root.mkdir(exist_ok=True)
candidates = list(data_root.rglob("metadata.csv"))
if not candidates:
    archive = ROOT / "waterbirds.tar.gz"
    for url in download_urls:
        try:
            run("curl", "--fail", "--location", "--retry", "2", "--connect-timeout", "20",
                "--output", archive, url)
            with tarfile.open(archive, "r:gz") as bundle:
                bundle.extractall(data_root, filter="data")
            break
        except (subprocess.CalledProcessError, tarfile.TarError) as error:
            print(f"Download indisponível nessa origem: {error}")
    candidates = list(data_root.rglob("metadata.csv"))
matches = [p.parent for p in candidates
           if hashlib.sha256(p.read_bytes()).hexdigest() == expected_metadata_digest]
if len(matches) != 1:
    raise RuntimeError("Não encontrei os metadados oficiais exatos. Confira o download do Waterbirds v1.0.")
DATA = matches[0]
run(PYTHON, "-m", "reproduction", "check", "--data", DATA)


In [ ]:
# Comece com [1]. Para o resultado final, use [1, 2, 3, 4, 5].
# Para executar só sementes que ainda faltam, use, por exemplo, [3, 4, 5].
SEMENTES = [1]

if not SEMENTES or any(s not in range(1, 6) for s in SEMENTES):
    raise ValueError("Indique uma ou mais sementes de 1 a 5.")

def executar(arquitetura):
    flags = [flag for s in sorted(set(SEMENTES)) for flag in ("--seed", str(s))]
    run(PYTHON, "-m", "reproduction", "run", "--data", DATA,
        "--architecture", arquitetura, *flags)


In [ ]:
# ShuffleNet V2 ×0,5
executar("shufflenet_v2_x0_5")


In [ ]:
# MobileNetV3-Small
executar("mobilenet_v3_small")


In [ ]:
# DenseNet-121
executar("densenet121")


In [ ]:
# EfficientNet-B0
executar("efficientnet_b0")


In [ ]:
# Baixar os resultados e registros disponíveis nesta sessão.
from google.colab import files

arquiteturas = ('shufflenet_v2_x0_5', 'mobilenet_v3_small', 'densenet121', 'efficientnet_b0',)
pastas = [
    ROOT / "runs" / modelo for modelo in arquiteturas
    if (ROOT / "runs" / modelo / "results.csv").is_file()
]
if not pastas:
    raise RuntimeError("Nenhuma arquitetura terminou uma semente.")

completo = len(pastas) == len(arquiteturas) and all(
    (pasta / "summary.csv").is_file() for pasta in pastas
)
sufixo = "FINAL" if completo else "PARCIAL"
destino = ROOT / f"Waterbirds_Passo2_{sufixo}.zip"
arquivos_raiz = ("protocol.json", "results.csv", "summary.csv")
arquivos_semente = ("raw.json", "evaluation.log", "commands.txt")

with zipfile.ZipFile(destino, "w", compression=zipfile.ZIP_DEFLATED) as pacote:
    for pasta in pastas:
        concluidas = [p for p in sorted(pasta.glob("seed-*")) if (p / "raw.json").is_file()]
        print(f"{pasta.name}: {len(concluidas)} de 5 sementes concluídas nesta sessão")
        candidatos = [pasta / nome for nome in arquivos_raiz]
        for semente in concluidas:
            candidatos.extend(semente / nome for nome in arquivos_semente)
            candidatos.extend(semente / "erm" / nome for nome in ("args.json", "log.txt"))
        for item in candidatos:
            if item.is_file():
                pacote.write(item, item.relative_to(ROOT))
files.download(str(destino))
